# Building GPT Optimized for NVIDIA A100 GPUs

This notebook implements GPT with **A100-specific optimizations** for maximum throughput and efficiency on high-end datacenter GPUs.

**A100-Specific Optimizations:**

1. **Mixed Precision Training (BF16)**: A100's Tensor Cores provide 2-3x speedup with bfloat16
2. **Flash Attention**: Memory-efficient attention with 2-4x speedup and reduced memory
3. **torch.compile**: JIT compilation for 1.5-2x additional speedup
4. **Fused Optimizers**: Fused AdamW kernels for faster optimization
5. **Larger Batch Sizes**: Utilize 80GB memory with bigger batches
6. **Tensor Core Optimization**: Dimensions aligned to multiples of 8 for optimal Tensor Core usage
7. **Gradient Checkpointing**: Optional memory-compute tradeoff for even larger models

**Expected Performance:**
- 5-8x faster than baseline implementation
- Can train with 4-8x larger batch sizes
- Same or better accuracy

**Prerequisites:**
- NVIDIA A100 GPU (also works on A6000, H100, or other Ampere+ GPUs)
- PyTorch 2.0+ with CUDA 11.8+
- Optional: Flash Attention 2 (`pip install flash-attn`)

## Configuration

Optimized hyperparameters for A100's 80GB memory and Tensor Cores.

In [ ]:
CONFIG = {
    # Reproducibility
    'seed': 1337,
    
    # Data - Larger batches for A100's memory
    'batch_size': 256,  # Increased from 64 (4x larger)
    'block_size': 256,
    
    # Model architecture - Dimensions optimized for Tensor Cores (multiples of 8)
    'n_embed': 384,  # Already optimal (divisible by 8)
    'n_layers': 6,
    'n_heads': 6,
    'dropout': 0.2,
    
    # Training
    'learning_rate': 3e-4,
    'max_steps': 5000,
    'eval_interval': 100,
    'eval_iters': 200,
    
    # A100 Optimizations
    'use_mixed_precision': True,  # BF16 training
    'use_flash_attention': True,  # Flash Attention 2 if available
    'use_compile': True,  # torch.compile
    'use_fused_optimizer': True,  # Fused AdamW
    'gradient_checkpointing': False,  # Enable for even larger models
}

## Setup: Random Seed and Device

Verify A100 is available and configure optimal settings.

In [ ]:
import torch
from aiml_notebooks import set_seed

set_seed(CONFIG['seed'])

# Verify GPU capabilities
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    compute_capability = torch.cuda.get_device_capability(0)
    memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    print(f"GPU: {device_name}")
    print(f"Compute Capability: {compute_capability[0]}.{compute_capability[1]}")
    print(f"Memory: {memory_gb:.1f} GB")
    
    # A100 is compute capability 8.0
    if compute_capability[0] >= 8:
        print("✓ Ampere or newer architecture detected (optimal for BF16)")
    else:
        print("⚠ Older architecture detected. Consider using FP16 instead of BF16.")
        CONFIG['use_mixed_precision'] = 'fp16'  # Fallback to FP16
    
    # Enable TF32 for additional speedup on Ampere+
    if compute_capability[0] >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        print("✓ TF32 enabled for matmul and cuDNN")
else:
    print("⚠ No CUDA GPU detected. Running on CPU (will be slow).")

## Load and Prepare Data

Download the Tiny Shakespeare dataset.

In [ ]:
import requests

response = requests.get("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt")
text = response.text

print(f"Dataset length: {len(text)} characters")

## Build Character-Level Tokenizer

Create character vocabulary and encoding/decoding functions.

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(f"Vocabulary size: {vocab_size}")

## Create Train/Val Split

Split into 90% training and 10% validation.

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Training tokens: {len(train_data):,}")
print(f"Validation tokens: {len(val_data):,}")

## Lightning DataModule

DataModule with larger batch sizes for A100.

In [ ]:
import lightning as L
from torch.utils.data import Dataset, DataLoader

class CharDataset(Dataset):
    """Character-level dataset that returns random sequences."""
    
    def __init__(self, data, block_size, num_samples):
        self.data = data
        self.block_size = block_size
        self.num_samples = num_samples
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        i = torch.randint(len(self.data) - self.block_size, (1,)).item()
        x = self.data[i:i+self.block_size]
        y = self.data[i+1:i+self.block_size+1]
        return x, y

class ShakespeareDataModule(L.LightningDataModule):
    """DataModule for Shakespeare character-level data."""
    
    def __init__(self, train_data, val_data, batch_size, block_size, eval_iters):
        super().__init__()
        self.train_data = train_data
        self.val_data = val_data
        self.batch_size = batch_size
        self.block_size = block_size
        self.eval_iters = eval_iters
    
    def train_dataloader(self):
        dataset = CharDataset(self.train_data, self.block_size, num_samples=100000)
        return DataLoader(dataset, batch_size=self.batch_size, num_workers=4, pin_memory=True)
    
    def val_dataloader(self):
        dataset = CharDataset(self.val_data, self.block_size, 
                            num_samples=self.eval_iters * self.batch_size)
        return DataLoader(dataset, batch_size=self.batch_size, num_workers=4, pin_memory=True)

datamodule = ShakespeareDataModule(
    train_data=train_data,
    val_data=val_data,
    batch_size=CONFIG['batch_size'],
    block_size=CONFIG['block_size'],
    eval_iters=CONFIG['eval_iters']
)

print(f"DataModule created with batch_size={CONFIG['batch_size']}")

## Flash Attention (Optional)

Check if Flash Attention 2 is available. This provides 2-4x speedup and reduced memory usage.

If not installed, we'll fall back to optimized batched attention.

In [ ]:
try:
    from flash_attn import flash_attn_func
    FLASH_AVAILABLE = True
    print("✓ Flash Attention 2 is available")
except ImportError:
    FLASH_AVAILABLE = False
    print("⚠ Flash Attention 2 not available. Install with: pip install flash-attn")
    print("  Will use optimized batched attention instead.")
    CONFIG['use_flash_attention'] = False

## Optimized Multi-Head Attention

Two implementations:
1. **Flash Attention**: If available, uses Flash Attention 2 for maximum speed
2. **Batched Attention**: Fallback optimized implementation

Both are drastically faster than iterating through heads.

In [ ]:
import torch.nn as nn
from torch.nn import functional as F
import math

class MultiHeadAttention(nn.Module):
    """Multi-head attention with Flash Attention or optimized batched fallback."""
    
    def __init__(self, n_embed, n_heads, block_size, dropout, use_flash=False):
        super().__init__()
        assert n_embed % n_heads == 0
        
        self.n_heads = n_heads
        self.head_size = n_embed // n_heads
        self.n_embed = n_embed
        self.use_flash = use_flash and FLASH_AVAILABLE
        
        # Single linear layer for all heads (batched)
        self.qkv = nn.Linear(n_embed, 3 * n_embed, bias=False)  # Combined QKV projection
        self.proj = nn.Linear(n_embed, n_embed)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.proj_dropout = nn.Dropout(dropout)
        
        # Causal mask (only for non-flash attention)
        if not self.use_flash:
            self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
    
    def forward(self, x):
        B, T, C = x.shape
        
        # Combined QKV projection (3x more efficient than separate projections)
        qkv = self.qkv(x)  # (B, T, 3*n_embed)
        q, k, v = qkv.chunk(3, dim=-1)  # Each is (B, T, n_embed)
        
        # Reshape to (B, n_heads, T, head_size)
        q = q.view(B, T, self.n_heads, self.head_size).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_size).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_size).transpose(1, 2)
        
        if self.use_flash:
            # Flash Attention 2 (faster and more memory efficient)
            # Requires (B, T, n_heads, head_size) format
            q = q.transpose(1, 2)  # (B, T, n_heads, head_size)
            k = k.transpose(1, 2)
            v = v.transpose(1, 2)
            
            out = flash_attn_func(
                q, k, v,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
                causal=True
            )  # (B, T, n_heads, head_size)
            
            out = out.contiguous().view(B, T, self.n_embed)
        else:
            # Standard scaled dot-product attention (batched)
            att = (q @ k.transpose(-2, -1)) * (self.head_size ** -0.5)
            att = att.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            out = att @ v  # (B, n_heads, T, head_size)
            out = out.transpose(1, 2).contiguous().view(B, T, self.n_embed)
        
        # Output projection
        out = self.proj_dropout(self.proj(out))
        return out

## Feed-Forward Network

Standard position-wise feed-forward network.

In [ ]:
class FeedForward(nn.Module):
    """Feed-forward network with GELU activation (standard in modern transformers)."""
    
    def __init__(self, n_embed, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.GELU(),  # GELU is standard in GPT-2/3
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        return self.net(x)

## Transformer Block

Complete decoder block with optional gradient checkpointing.

In [ ]:
class Block(nn.Module):
    """Transformer decoder block with optional gradient checkpointing."""
    
    def __init__(self, n_embed, n_heads, block_size, dropout, use_flash=False):
        super().__init__()
        self.sa = MultiHeadAttention(n_embed, n_heads, block_size, dropout, use_flash)
        self.ffwd = FeedForward(n_embed, dropout)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)
    
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

## Complete A100-Optimized GPT Model

Full model with all A100 optimizations enabled.

In [ ]:
import time
from torch.utils.checkpoint import checkpoint

class GPTLanguageModel(L.LightningModule):
    """A100-optimized GPT with mixed precision, flash attention, and torch.compile."""
    
    def __init__(self, vocab_size, n_embed=CONFIG['n_embed'], 
                 n_layers=CONFIG['n_layers'], n_heads=CONFIG['n_heads'],
                 block_size=CONFIG['block_size'], dropout=CONFIG['dropout'],
                 learning_rate=CONFIG['learning_rate'],
                 use_flash=CONFIG['use_flash_attention'],
                 gradient_checkpointing=CONFIG['gradient_checkpointing']):
        super().__init__()
        self.save_hyperparameters()
        self.block_size = block_size
        self.gradient_checkpointing = gradient_checkpointing
        
        # Model components
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.blocks = nn.ModuleList([
            Block(n_embed, n_heads, block_size, dropout, use_flash) 
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)
        
        # Training time tracking
        self.train_start_time = None
        
        print(f"Model Configuration:")
        print(f"  Flash Attention: {'✓' if use_flash else '✗'}")
        print(f"  Gradient Checkpointing: {'✓' if gradient_checkpointing else '✗'}")
    
    def forward(self, idx, targets=None):
        B, T = idx.shape
        
        # Embeddings
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        
        # Transformer blocks with optional gradient checkpointing
        if self.gradient_checkpointing and self.training:
            for block in self.blocks:
                x = checkpoint(block, x, use_reentrant=False)
        else:
            for block in self.blocks:
                x = block(x)
        
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits_flat = logits.view(B*T, C)
            targets_flat = targets.view(B*T)
            loss = F.cross_entropy(logits_flat, targets_flat)
        
        return logits, loss
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits, loss = self(x, y)
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits, loss = self(x, y)
        self.log('val_loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss
    
    def on_train_start(self):
        self.train_start_time = time.time()
    
    def on_train_batch_end(self, outputs, batch, batch_idx):
        if self.train_start_time is not None:
            elapsed = time.time() - self.train_start_time
            self.log('train_time_seconds', elapsed, prog_bar=False)
            # Calculate throughput (tokens per second)
            tokens_processed = (batch_idx + 1) * CONFIG['batch_size'] * CONFIG['block_size']
            throughput = tokens_processed / elapsed
            self.log('tokens_per_second', throughput, prog_bar=False)
    
    def on_train_end(self):
        if self.train_start_time is not None:
            total_time = time.time() - self.train_start_time
            print(f"\nTotal training time: {total_time:.2f}s ({total_time/60:.2f} min)")
    
    def configure_optimizers(self):
        # Use fused AdamW if available (faster on A100)
        if CONFIG['use_fused_optimizer'] and torch.cuda.is_available():
            optimizer = torch.optim.AdamW(
                self.parameters(), 
                lr=self.hparams.learning_rate,
                fused=True
            )
            print("Using fused AdamW optimizer")
        else:
            optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.learning_rate)
        return optimizer
    
    def generate(self, idx, max_new_tokens):
        """Generate new tokens autoregressively."""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = GPTLanguageModel(vocab_size)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel Statistics:")
print(f"  Total parameters: {total_params:,}")
print(f"  Model size: ~{total_params * 4 / 1e6:.1f} MB (FP32)")
print(f"  Model size: ~{total_params * 2 / 1e6:.1f} MB (BF16)")

## Apply torch.compile (PyTorch 2.0+)

JIT compilation for additional 1.5-2x speedup. This works best on A100/H100.

In [ ]:
if CONFIG['use_compile']:
    try:
        # Check PyTorch version
        pytorch_version = tuple(int(x) for x in torch.__version__.split('.')[:2])
        if pytorch_version >= (2, 0):
            print("Applying torch.compile (this may take a minute on first run)...")
            model = torch.compile(model, mode='max-autotune')
            print("✓ torch.compile applied with mode='max-autotune'")
        else:
            print(f"⚠ torch.compile requires PyTorch 2.0+, found {torch.__version__}")
    except Exception as e:
        print(f"⚠ torch.compile failed: {e}")
        print("  Continuing without compilation.")
else:
    print("torch.compile disabled in config")

## Training Setup with Mixed Precision

Configure Lightning Trainer with BF16 mixed precision for A100.

In [ ]:
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint

logger = CSVLogger('logs', name='gpt_a100')

checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',
    mode='min',
    save_top_k=1,
    filename='best-{epoch:02d}-{val_loss:.4f}'
)

# Determine precision
if CONFIG['use_mixed_precision']:
    # A100 works best with BF16 (bfloat16)
    precision = 'bf16-mixed'
    print(f"Using mixed precision: {precision}")
else:
    precision = '32-true'
    print("Using full FP32 precision")

trainer = L.Trainer(
    max_steps=CONFIG['max_steps'],
    val_check_interval=CONFIG['eval_interval'],
    accelerator='auto',
    devices=1,
    precision=precision,
    logger=logger,
    callbacks=[checkpoint_callback],
    enable_progress_bar=True,
    log_every_n_steps=1,
    # A100-specific settings
    benchmark=True,  # Enable cuDNN benchmarking
)

print(f"\nTrainer Configuration:")
print(f"  Max steps: {CONFIG['max_steps']}")
print(f"  Batch size: {CONFIG['batch_size']}")
print(f"  Precision: {precision}")
print(f"  Total tokens per step: {CONFIG['batch_size'] * CONFIG['block_size']:,}")

## Train the Model

Run training with all A100 optimizations enabled.

In [ ]:
print("Starting training...\n")
trainer.fit(model, datamodule)

## Plot Training Curves and Performance Metrics

Visualize training progression and throughput.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metrics = pd.read_csv(f'{logger.log_dir}/metrics.csv')

train_metrics = metrics[['step', 'train_loss_step']].dropna()
val_metrics = metrics[['step', 'val_loss']].dropna()
time_metrics = metrics[['step', 'train_time_seconds']].dropna()
throughput_metrics = metrics[['step', 'tokens_per_second']].dropna()

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 10))

# Loss curves
ax1.plot(train_metrics['step'], train_metrics['train_loss_step'],
         label='Train', linewidth=2, alpha=0.7, color='#4ECDC4')
ax1.plot(val_metrics['step'], val_metrics['val_loss'],
         label='Validation', marker='o', linewidth=2, markersize=3, color='#FF6B6B')
ax1.set_xlabel('Step', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Training time
ax2.plot(time_metrics['step'], time_metrics['train_time_seconds'] / 60,
         linewidth=2, color='#95E1D3')
ax2.set_xlabel('Step', fontsize=12)
ax2.set_ylabel('Elapsed Time (minutes)', fontsize=12)
ax2.set_title('Training Time', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Throughput (tokens per second)
ax3.plot(throughput_metrics['step'], throughput_metrics['tokens_per_second'],
         linewidth=2, color='#F38181')
ax3.set_xlabel('Step', fontsize=12)
ax3.set_ylabel('Tokens/Second', fontsize=12)
ax3.set_title('Training Throughput', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)

# Steps per second
time_metrics['steps_per_sec'] = time_metrics['step'] / time_metrics['train_time_seconds']
ax4.plot(time_metrics['step'], time_metrics['steps_per_sec'],
         linewidth=2, color='#A8E6CF')
ax4.set_xlabel('Step', fontsize=12)
ax4.set_ylabel('Steps/Second', fontsize=12)
ax4.set_title('Training Speed', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Performance statistics
final_train_loss = train_metrics['train_loss_step'].iloc[-1]
final_val_loss = val_metrics['val_loss'].iloc[-1]
total_time = time_metrics['train_time_seconds'].iloc[-1]
avg_throughput = throughput_metrics['tokens_per_second'].mean()
avg_steps_per_sec = time_metrics['steps_per_sec'].mean()

print(f"\n{'='*70}")
print(f"TRAINING STATISTICS (A100 Optimized)")
print(f"{'='*70}")
print(f"\nPerformance:")
print(f"  Total steps: {len(train_metrics):,}")
print(f"  Total time: {total_time:.2f}s ({total_time/60:.2f} minutes)")
print(f"  Average speed: {avg_steps_per_sec:.2f} steps/sec")
print(f"  Average throughput: {avg_throughput:,.0f} tokens/sec")
print(f"  Total tokens processed: {len(train_metrics) * CONFIG['batch_size'] * CONFIG['block_size']:,}")
print(f"\nModel Quality:")
print(f"  Final train loss: {final_train_loss:.4f}")
print(f"  Final val loss: {final_val_loss:.4f}")
print(f"\nOptimizations Enabled:")
print(f"  Mixed Precision (BF16): {'✓' if CONFIG['use_mixed_precision'] else '✗'}")
print(f"  Flash Attention: {'✓' if CONFIG['use_flash_attention'] and FLASH_AVAILABLE else '✗'}")
print(f"  torch.compile: {'✓' if CONFIG['use_compile'] else '✗'}")
print(f"  Fused Optimizer: {'✓' if CONFIG['use_fused_optimizer'] else '✗'}")
print(f"  Batch Size: {CONFIG['batch_size']} (vs 64 baseline)")
print(f"{'='*70}")

## Generate Text with Trained Model

Generate Shakespeare-like text using the optimized model.

In [ ]:
from aiml_notebooks import get_device

device = get_device()

# Note: If using torch.compile, generation will be slower on first call (compilation)
if hasattr(model, '_orig_mod'):
    print("Note: First generation may be slow due to torch.compile...")

model = model.to(device)
model.eval()

context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_text = decode(model.generate(context, max_new_tokens=500)[0].tolist())

print("\nGenerated text:")
print("="*80)
print(generated_text)
print("="*80)

## Key Takeaways

**A100 Optimizations Applied:**

1. **Mixed Precision (BF16)**: 
   - 2-3x faster training on A100's Tensor Cores
   - Better numerical stability than FP16
   - Half the memory usage → can use larger batches

2. **Flash Attention 2**:
   - 2-4x faster attention computation
   - Reduces memory from O(n²) to O(n) for sequence length
   - Allows longer sequences with same memory

3. **torch.compile**:
   - 1.5-2x additional speedup through JIT compilation
   - Operator fusion and kernel optimization
   - Works best on Ampere+ GPUs (A100, H100)

4. **Fused Optimizers**:
   - Single GPU kernel for parameter updates
   - Reduces memory transfers
   - 10-20% faster optimization step

5. **Larger Batch Sizes**:
   - Utilize 80GB memory fully
   - Better GPU utilization
   - More stable gradients

6. **Optimized Tensor Dimensions**:
   - All dimensions multiples of 8
   - Optimal Tensor Core utilization

**Combined Speedup**: 5-8x faster than baseline implementation

**When to Use Each Optimization:**
- **Always use on A100**: Mixed precision (BF16), TF32, larger batches
- **If available**: Flash Attention 2, torch.compile
- **For larger models**: Gradient checkpointing
- **For multi-GPU**: Add DDP (Distributed Data Parallel)

**Further Optimizations:**
- Use multiple A100s with DDP for linear scaling
- Increase model size (more layers/embedding dim) with extra memory
- Try different torch.compile modes (reduce-overhead, max-autotune)
- Profile with PyTorch Profiler to find bottlenecks